# Delay Comparison
Compare delay distributions across bus, streetcar and subway on raw data, to justify excluding subway.

In [12]:
import sys
from pathlib import Path
import pandas as pd
# Read the src module
repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

from src.data.load import load_data, load_subway

In [13]:
# Load data for bus, streetcar, and subway
surface = load_data()
subway = load_subway()

df = pd.concat([surface, subway], ignore_index=True)
df.shape

(1072754, 11)

In [14]:
# Check to see if we have any N/A values for min_delay
print(df['min_delay'].isna().sum())

# Drop the n/a values row
df = df.dropna(subset=["min_delay"])

# Print the summary of the table
# pct_zero: share of incidents with a 0-minute delay. Median alone only says "at least half are 0";
#           this shows how many. A mostly-zero target is a different prediction problem.
# p95:      95th percentile. Shows how long typical long delays are, without being distorted
#           by extreme outliers (e.g. a 246,245-minute bus record) the way the mean is.
summary = df.groupby("mode").agg(
    num_rows = ("min_delay", "count"),
    median = ("min_delay", "median"),
    mean        = ("min_delay", "mean"),
    pct_zero    = ("min_delay", lambda x: (x == 0).mean() * 100),
    p95         = ("min_delay", lambda x: x.quantile(0.95)),
    num_incidents = ("incident", "nunique"),   
) 

summary.round(1)

542


,num_rows,median,mean,pct_zero,p95,num_incidents
mode,,,,,,
bus,703147,10.0,20.2,3.2,30.0,30
streetcar,146901,8.0,13.7,5.2,40.0,22
subway,222164,0.0,2.5,65.2,10.0,222


According to the table, we can see that most subway incidents cause no delay, and the subway also uses a completely different set of incident categories (refer to the code description in [data](../data/raw/csv/ttc-subway-delay-codes.csv)). 

The table displays the number of rows, median, mean, percentage of 0, 95th percentile, and number of incidents of bus, streetcar, and subway with the 542 rows missing a delay removed. Bus and streetcar delays are around 10 and 8 minutes, and only 3.2% and 5.2% record a 0-minute delay. 

As we can see from the median column, most of the subway incidents cause 0 delay, and 65.2% of incidents record a 0-minute delay. Therefore, we'll focus on the min_delay values of buses and streetcars for training models. Subway is a different prediction problem (whether there's a delay, not how long).